# User-friendly direct-CP quickstart

The recommended compact CP workflow: shared `CPRealImag` parameters → `generate_cp_toy` → `CPFitSession` → `report()` → charge-separated projections. Toy generation uses the default inverse-transform sampler.


In [ ]:
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    CPFitSession,CPRealImag,DecayChannel,DecayModel,NonResonant,Parameter,
    RealImag,Resonance,enable_x64,generate_cp_toy,plot_dalitz,
)
enable_x64()


In [ ]:
cp=CPRealImag(
    Parameter.coefficient("NR.x",0.8,bounds=(-2,2),owner="NR",step=0.02),
    Parameter.coefficient("NR.y",0.2,bounds=(-2,2),owner="NR",step=0.02),
    Parameter.coefficient("NR.dx",0.08,bounds=(-0.4,0.4),owner="NR",step=0.01),
    Parameter.coefficient("NR.dy",-0.05,bounds=(-0.4,0.4),owner="NR",step=0.01),
)
def make(parent,final,q):
    return DecayModel(
        DecayChannel(parent,final),
        [
            Resonance("Kstar",(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),
            NonResonant(cp.for_charge(q)),
        ],
        normalization_method="square-dalitz",normalization_resolution=180,normalization_pair=(0,2),
    )
plus_model=make("B+",("K+","pi+","pi-"),+1)
minus_model=make("B-",("K-","pi-","pi+"),-1)
truth={p.name:p.value for p in plus_model.parameters}

plus_data,minus_data=generate_cp_toy(
    plus_model,minus_model,30_000,parameters=truth,seed=1717
)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
plot_dalitz(plus_data,x="s13",y="s23",ax=axes[0],title="B+")
plot_dalitz(minus_data,x="s13",y="s23",ax=axes[1],title="B-")
plt.show()

session=CPFitSession(plus_model,minus_model,plus_data,minus_data)
start={p.name:p.value+0.05 for p in session.parameters if not p.fixed}
result=session.fit(start,simplex=True,ncall=40_000)
session.report(result)
session.plot_projection(result,"s13")
plt.show()
